In [ ]:
"""
Equity-Curve-Driven Prop Firm Lifecycle Simulator
===================================================
Instead of generating trades from win-rate/expectancy, this walks a REAL
(or resampled) trade-by-trade equity curve through the full lifecycle:

    challenge -> pass? -> funded -> profit hits $2000 -> withdraw $1000
                       -> fail (drawdown) -> buy new account ($100) -> repeat

Reports: accounts bought, accounts passed, total withdrawn, total spent, net profit.
"""

import numpy as np
import pandas as pd

# ----------------------------
# 1. PARAMETERS  <-- edit to match your real setup
# ----------------------------
ACCOUNT_SIZE       = 50_000
PROFIT_TARGET       = 3_000     # challenge phase target
MAX_DRAWDOWN        = 2_000     # trailing-after-day drawdown, both phases
ACCOUNT_COST        = 100       # $ cost to buy/reset a challenge account
WITHDRAW_THRESHOLD  = 2_000     # $ profit in funded account before a withdrawal triggers
WITHDRAW_AMOUNT     = 1_000     # $ pulled out per threshold hit
TRADES_PER_DAY      = 3         # ASSUMPTION for day-close trailing DD - edit to match your data
CYCLE_EQUITY_CURVE  = True      # if the curve runs out, loop back to the start (bootstrap-style)


# ----------------------------
# 2. LOAD YOUR EQUITY CURVE
#    Replace this with: pd.read_csv("my_trades.csv")["pnl"].values
#    File should have one row per trade, a column of $ PnL per trade.
# ----------------------------
def load_equity_curve_demo(n_trades=3000, seed=42):
    """DEMO ONLY: synthetic trade PnL series matching 35% WR / 0.02R expectancy / $100 risk."""
    rng = np.random.default_rng(seed)
    win_rate, expectancy_R, loss_R, risk = 0.35, 0.02, 1.0, 100
    win_R = (expectancy_R + (1 - win_rate) * loss_R) / win_rate
    wins = rng.random(n_trades) < win_rate
    pnl = np.where(wins, risk * win_R, -risk * loss_R)
    return pnl

equity_curve = load_equity_curve_demo()
print(f"Loaded equity curve with {len(equity_curve)} trades "
      f"(sum PnL if traded once through: ${equity_curve.sum():,.0f})\n")


# ----------------------------
# 3. WALK THE CURVE THROUGH THE FULL LIFECYCLE
# ----------------------------
def run_lifecycle(curve, cycle=CYCLE_EQUITY_CURVE, max_iterations=2_000_000):
    idx = 0
    n = len(curve)

    accounts_bought = 0
    accounts_passed = 0
    total_withdrawn = 0.0
    withdrawal_events = 0

    # start first account
    phase = "challenge"
    balance = ACCOUNT_SIZE
    peak_close = ACCOUNT_SIZE
    day_trade_count = 0
    funded_reference_balance = None  # balance right after funding / last withdrawal

    accounts_bought += 1  # buying the very first account

    iterations = 0
    while iterations < max_iterations:
        iterations += 1

        if idx >= n:
            if cycle:
                idx = 0
            else:
                break

        pnl = curve[idx]
        idx += 1

        balance += pnl
        day_trade_count += 1

        # --- check drawdown floor (trailing-after-day) ---
        floor = peak_close - MAX_DRAWDOWN
        if balance <= floor:
            # account blown -> buy a new one, restart challenge phase
            accounts_bought += 1
            phase = "challenge"
            balance = ACCOUNT_SIZE
            peak_close = ACCOUNT_SIZE
            day_trade_count = 0
            funded_reference_balance = None
            continue

        # --- challenge phase: check profit target ---
        if phase == "challenge" and balance >= ACCOUNT_SIZE + PROFIT_TARGET:
            accounts_passed += 1
            phase = "funded"
            funded_reference_balance = balance  # profit tracking starts fresh here

        # --- funded phase: check withdrawal threshold (loop in case of a big jump) ---
        if phase == "funded":
            while balance - funded_reference_balance >= WITHDRAW_THRESHOLD:
                balance -= WITHDRAW_AMOUNT
                total_withdrawn += WITHDRAW_AMOUNT
                withdrawal_events += 1
                peak_close -= WITHDRAW_AMOUNT  # real cash left -> floor drops too
                funded_reference_balance = balance

        # --- end of trading day: lock in new high-water mark if applicable ---
        if day_trade_count == TRADES_PER_DAY:
            peak_close = max(peak_close, balance)
            day_trade_count = 0

        # stop condition: curve exhausted and not cycling
        if idx >= n and not cycle:
            break

    total_cost = accounts_bought * ACCOUNT_COST
    net_profit = total_withdrawn - total_cost

    return {
        "accounts_bought": accounts_bought,
        "accounts_passed": accounts_passed,
        "withdrawal_events": withdrawal_events,
        "total_withdrawn": total_withdrawn,
        "total_cost": total_cost,
        "net_profit": net_profit,
        "final_phase": phase,
        "final_balance": balance,
    }


# ----------------------------
# 4. BOOTSTRAP MONTE CARLO
#    Resamples trades (with replacement) from your equity curve's PnL
#    distribution to get statistically robust expected totals, instead of
#    relying on the one exact historical ordering.
# ----------------------------
def bootstrap_monte_carlo(curve, trade_horizon=3000, n_sims=500, seed=0):
    rng = np.random.default_rng(seed)
    rows = []
    for _ in range(n_sims):
        sample = rng.choice(curve, size=trade_horizon, replace=True)
        res = run_lifecycle(sample, cycle=False)
        rows.append(res)
    return pd.DataFrame(rows)


if __name__ == "__main__":
    print("=== A) Single walk through your actual trade history (no extrapolation) ===")
    result = run_lifecycle(equity_curve, cycle=False)
    for k, v in result.items():
        print(f"{k:>20}: {'$'+format(v,',.0f') if isinstance(v, float) else v}")
    pass_rate = result["accounts_passed"] / result["accounts_bought"] if result["accounts_bought"] else 0
    print(f"\nPass rate (per account attempt): {pass_rate:.1%}")
    print(f"Net profit after account costs:  ${result['net_profit']:,.0f}")

    print("\n=== B) Bootstrap Monte Carlo (500 resampled histories, 3000 trades each) ===")
    df = bootstrap_monte_carlo(equity_curve, trade_horizon=3000, n_sims=500)
    summary = df[["accounts_bought", "accounts_passed", "withdrawal_events",
                   "total_withdrawn", "total_cost", "net_profit"]].agg(["mean", "median", "std"])
    print(summary.round(0))

    df["pass_rate"] = df["accounts_passed"] / df["accounts_bought"]
    print(f"\nMean pass rate per account attempt: {df['pass_rate'].mean():.1%}")
    print(f"5th / 95th percentile net profit:   "
          f"${df['net_profit'].quantile(0.05):,.0f} / ${df['net_profit'].quantile(0.95):,.0f}")
    print(f"P(net profit > $0):                 {(df['net_profit'] > 0).mean():.1%}")